<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/mean_escape_time/ou_ball/ou_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ornstein–Uhlenbeck mean escape time: five-method comparison

This is the principal notebook for the OU experiment in the paper. It keeps the original problem, five neural formulations, Monte Carlo data term, adaptive loss weights, optimizer schedule, and error measures in one readable run.

**Suggested path:** read Sections 1–3, choose `RUN_SCALE`, prepare the shared data once, train the selected methods, then compare all results in the same figures and tables. `quick` is a workflow demonstration with the same shallow `tanh` network used by the committor notebook. `paper` restores the recorded OU architecture and numerical settings and is expensive.


## 1. Problem and reference solution

In the disk $\Omega=B_R(0)\subset\mathbb R^2$, consider

$$dX_t=-\theta X_t\,dt+\sigma\,dB_t,$$

and the first exit time $T=\inf\{t\geq0:X_t\in\partial\Omega\}$. Its mean $\tau(x)=\mathbb E_x[T]$ solves

$$-\mathcal L\tau=1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

with $\mathcal L u=-\theta x\cdot\nabla u+\frac{\sigma^2}{2}\Delta u$.

The recorded experiment uses $R=2$, $\theta=3$, and **$\sigma^2=2$**. For $d$ dimensions, Kersting et al. give

$$
\tau(x)=\frac{1}{\lambda^{d/2}\sigma^2}
\int_{\lVert x\rVert}^{R}z^{1-d}e^{\lambda z^2}
\gamma\!\left(\frac d2,\lambda z^2\right)\,dz,
\qquad \lambda=\frac{\theta}{\sigma^2},
$$

where $\gamma(s,y)=\int_0^y t^{s-1}e^{-t}\,dt$. In two dimensions this becomes

$$
\tau(x)=\frac1\theta\int_{\lVert x\rVert}^{R}
\frac{e^{\lambda z^2}-1}{z}\,dz.
$$

The notebook evaluates this integral for the reference field. It also generates shared Euler–Maruyama labels because that data term was part of every original training objective.


## 2. The five methods

The default `quick` run uses the committor notebook's shallow network: two width-64 hidden layers with `tanh` activations. The `paper` preset restores the recorded width-30, three-hidden-layer OU network and Xavier initialization. In Section 5, the base network is an ordinary visible `nn.Module`; readers may replace it with any architecture that maps an `[n, 2]` coordinate tensor to one scalar per point. The five methods wrap that same base model with different boundary treatments and losses. The standard PINN provides the sampled-boundary baseline; the variational PINN provides a weak-form alternative; and the three exact-boundary powers show how the choice of boundary factor affects accuracy.

| Name in code | Trial function | Loss components |
|---|---|---|
| `standard` | $N_\theta$ | strong PDE, sampled boundary, Monte Carlo data |
| `adapted` | $dN_\theta$ | strong PDE, Monte Carlo data |
| `over` | $d^{3/2}N_\theta$ | strong PDE, Monte Carlo data |
| `under` | $d^{1/2}N_\theta$ | strong PDE, Monte Carlo data |
| `variational` | $N_\theta$ | weak residual, Nitsche boundary term, Monte Carlo data |

Here $d(x)=(R^2-\lVert x\rVert^2)/(2R)$ vanishes on the circle. The three exact-boundary methods use

$$u_\theta(x)=d(x)^\alpha N_\theta(x),\qquad
\alpha\in\{1,3/2,1/2\}.$$

The strong residual and Monte Carlo data losses are

$$J_{\rm pde}=\frac1{N_\Omega}\sum_i|\mathcal L u_\theta(x_i)+1|^2,
\qquad
J_{\rm data}=\frac1{N_d}\sum_j|u_\theta(z_j)-\widehat\tau_j|^2.$$

The standard PINN also uses $J_{\rm bc}=N_{\partial\Omega}^{-1}\sum_j|u_\theta(y_j)|^2$. Each active component $J_i$ is combined exactly as in the original notebooks,

$$J_{\rm train}=\sum_i e^{-2s_i}J_i+\sum_i s_i,$$

with trainable log scales $s_i$. Checkpoint selection now uses this same weighted objective. In the original OU variational notebook, the selection score counted the weak term twice and omitted the boundary and data terms. The correction changes which epoch is saved as best; it does not change the training objective.


In [ ]:
# Colab setup: this notebook works in Colab or a local Python environment.
import json
import math
import random
import secrets
import time
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate
import torch
from torch import nn

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

## 3. Editable configuration

Choose `RUN_MODE = "new"` for a fresh experiment. The notebook draws a seed when this cell runs and creates a new, uniquely named output folder; it never resumes or reuses an earlier run in this mode. To continue an interrupted run, set `RUN_MODE = "resume"` and paste its printed folder into `EXISTING_RUN_DIR`. To make plots from saved checkpoints, choose `"evaluate"` with the same folder. Resume and evaluation read the original seed and settings from that folder.

The `paper` preset uses the settings recorded in the original notebooks. The `quick` preset reduces the network and sample counts for a workflow check. In Colab, Drive is mounted only when saving to Drive is selected; outside Colab, output goes to `runs/ou` under the notebook directory.

Each completed run saves its configuration, shared Monte Carlo labels, evaluation points, model checkpoints, per-epoch training metrics, reference field, predictions, errors, and summary metrics. The solution and absolute-error comparison figures use one color range across their panels.

In [ ]:
RUN_SCALE = "quick"                 # "quick" or "paper"
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_MODE = "new"                     # "new", "resume", or "evaluate"
EXISTING_RUN_DIR = ""                # required for resume/evaluate; paste a printed run folder
RUN_TRAINING = False                 # set True to train the selected methods
RUN_SMOKE_CHECK = True               # one tiny forward/backward check for every method
RUN_LBFGS = False                    # paper workflow used Adam then L-BFGS
SAVE_TO_DRIVE = True

R = 2.0
theta = 3.0
sigma_squared = 2.0
sigma = math.sqrt(sigma_squared)

if RUN_SCALE == "paper":
    width, depth = 30, 3
    initialization_gain = 2.0
    pool_size, batch_size, adam_epochs = 8192, 512, 10_000
    data_anchors, paths_per_anchor = 256, 256
    mc_dt, mc_max_time = 1e-3, 20.0
    lbfgs_iterations = adam_epochs // 2
    report_every, checkpoint_every = 100, 500
elif RUN_SCALE == "quick":
    width, depth = 64, 2
    initialization_gain = None       # keep PyTorch's default Linear initialization
    pool_size, batch_size, adam_epochs = 512, 128, 40
    data_anchors, paths_per_anchor = 24, 24
    mc_dt, mc_max_time = 5e-3, 2.0
    lbfgs_iterations = 20
    report_every, checkpoint_every = 10, 10
else:
    raise ValueError("RUN_SCALE must be 'quick' or 'paper'")

learning_rate = 5e-4
number_of_decays = 100
final_learning_rate_factor = 1e-3

if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'")

if drive is not None and SAVE_TO_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not available after the mount attempt.")
    output_base = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/ou")
else:
    output_base = Path("runs/ou")
output_base.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "new":
    SEED = secrets.randbelow(2**32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_root = output_base / f"ou_{RUN_SCALE}_{run_stamp}_{secrets.token_hex(4)}"
    output_root.mkdir(parents=True, exist_ok=False)
    RESUME = False
    LOAD_SAVED_MODELS = False
else:
    if not EXISTING_RUN_DIR.strip():
        raise ValueError("Paste the exact printed run folder into EXISTING_RUN_DIR.")
    output_root = Path(EXISTING_RUN_DIR).expanduser()
    if not output_root.is_dir():
        raise FileNotFoundError(f"Run folder does not exist: {output_root}")
    run_config_path = output_root / "run_config.json"
    if run_config_path.is_file():
        saved_run = json.loads(run_config_path.read_text(encoding="utf-8"))
        SEED = int(saved_run["seed"])
    else:
        # Read older runs made before each run had its own manifest.
        legacy_data_path = output_root / f"ou_{RUN_SCALE}_shared_data.pt"
        if not legacy_data_path.is_file():
            raise FileNotFoundError(
                f"No run_config.json or legacy shared data file was found in {output_root}."
            )
        legacy_shared = torch.load(legacy_data_path, map_location="cpu", weights_only=False)
        SEED = int(legacy_shared["settings"]["seed"])
    RESUME = RUN_MODE == "resume"
    LOAD_SAVED_MODELS = RUN_MODE == "evaluate"

experiment_settings = {
    "run_scale": RUN_SCALE, "seed": SEED, "radius": R, "theta": theta,
    "sigma_squared": sigma_squared, "width": width, "depth": depth,
    "initialization_gain": initialization_gain, "pool_size": pool_size,
    "batch_size": batch_size, "adam_epochs": adam_epochs,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "learning_rate": learning_rate, "number_of_decays": number_of_decays,
    "final_learning_rate_factor": final_learning_rate_factor,
}

def require_matching_settings(record, expected, saved_path):
    if record.get("settings") != expected:
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Resume or evaluate using the original settings, or choose RUN_MODE='new'."
        )

run_config_path = output_root / "run_config.json"
if RUN_MODE == "new":
    run_config = {
        "schema_version": 1,
        "run_id": output_root.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "settings": experiment_settings,
        "methods": list(METHODS),
        "run_lbfgs": RUN_LBFGS,
    }
    with run_config_path.open("x", encoding="utf-8") as handle:
        json.dump(run_config, handle, indent=2)
        handle.write("\n")
else:
    if run_config_path.is_file():
        saved_run = json.loads(run_config_path.read_text(encoding="utf-8"))
        if saved_run.get("settings") != experiment_settings:
            raise ValueError("Selected settings do not match this run's saved configuration.")
        if RUN_MODE == "resume" and saved_run.get("methods") != list(METHODS):
            raise ValueError("Resume with the same METHODS list used by the saved run.")
        if RUN_MODE == "resume" and saved_run.get("run_lbfgs") != RUN_LBFGS:
            raise ValueError("Resume with the original RUN_LBFGS setting.")
        if RUN_MODE == "evaluate" and not set(METHODS).issubset(saved_run.get("methods", [])):
            raise ValueError("Evaluation METHODS must be present in the saved run.")
    print("Loaded seed from saved run:", SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print({"mode": RUN_MODE, "scale": RUN_SCALE, "seed": SEED, "output": str(output_root), "sigma_squared": sigma_squared})

### Reference solution

For dimension two, the lower incomplete-gamma expression simplifies to an integral with integrand $(e^{\lambda z^2}-1)/z$, where $\lambda=\theta/\sigma^2$. The special value at zero is taken by continuity. This is the same reference formula used by the original notebooks, evaluated more stably near the origin.


In [ ]:
def true_tau_radius(radius):
    lam = theta / sigma_squared

    def integrand(z):
        return 0.0 if z == 0.0 else np.expm1(lam * z * z) / z

    value, _ = integrate.quad(integrand, float(radius), R, epsabs=1e-10, epsrel=1e-10)
    return value / theta


def true_tau(points):
    radii = np.linalg.norm(np.asarray(points), axis=1)
    return np.array([true_tau_radius(r) for r in radii])


grid_n = 200
gx, gy = np.meshgrid(np.linspace(-R, R, grid_n), np.linspace(-R, R, grid_n))
disk_mask = gx**2 + gy**2 <= R**2
grid_points = np.column_stack((gx[disk_mask], gy[disk_mask]))
reference_grid = np.zeros_like(gx)
reference_grid[disk_mask] = true_tau(grid_points)

plt.figure(figsize=(5, 4))
plt.contourf(gx, gy, reference_grid, levels=50)
plt.colorbar(label=r"$\tau(x)$")
plt.gca().set_aspect("equal")
plt.title("OU reference mean escape time")
plt.show()


### A sample exit trajectory

The path below uses the same Euler–Maruyama update as the Monte Carlo labels. Its endpoint is the first discrete point outside the disk, or the final point at the time cap. The function takes the physical parameters as arguments, so it can also recreate the parameter-comparison trajectory plots from the earlier exploratory notebook without keeping that notebook.


In [ ]:
def simulate_ou_trajectory(
    start=(0.0, 0.0), *, theta_value=theta, sigma_squared_value=sigma_squared,
    radius_value=R, dt=1e-3, max_time=20.0, seed=SEED + 1,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = np.linalg.norm(point) >= radius_value
    for _ in range(int(max_time / dt)):
        if exited:
            break
        point = point - theta_value * point * dt + math.sqrt(sigma_squared_value * dt) * rng.standard_normal(2)
        path.append(point.copy())
        exited = np.linalg.norm(point) >= radius_value
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_ou_trajectory()
angle = np.linspace(0.0, 2.0 * np.pi, 300)
fig, axis = plt.subplots(figsize=(5, 5))
axis.plot(trajectory[:, 0], trajectory[:, 1], lw=1.0)
axis.plot(R * np.cos(angle), R * np.sin(angle), "k--", label="exit boundary")
axis.scatter(*trajectory[0], label="start")
axis.scatter(*trajectory[-1], marker="x", label="exit" if trajectory_exited else "time cap")
axis.set(aspect="equal", title=f"OU path: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
axis.legend()
plt.show()


## 4. Shared sampling and Monte Carlo labels

Points are uniform by area in the disk. From an anchor $z$, Euler–Maruyama advances

$$X_{n+1}=X_n-\theta X_n\,\Delta t+\sigma\sqrt{\Delta t}\,\xi_n,
\qquad \xi_n\sim\mathcal N(0,I_2),$$

until the path leaves the disk. The label $\widehat\tau(z)$ is the mean of the simulated exit times. Each label averages 256 paths in the paper setting. A path still inside at `mc_max_time` is recorded at that cap, matching the original implementation; the progress report and saved data state how many paths remain capped at each anchor.

The original one-method notebooks generated their Monte Carlo samples separately. This notebook generates one sample and reuses it for all five methods, changing the random realization but not the simulation rule, sample counts, or training losses.


In [ ]:
def sample_disk(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radius = R * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radius * torch.cos(angle), radius * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((R * torch.cos(angle), R * torch.sin(angle)), dim=1)


@torch.no_grad()
def monte_carlo_exit_time(x0, n_paths, dt, max_time):
    x0 = x0.to(DEVICE)
    batch = x0.shape[0]
    positions = x0[:, None, :].expand(batch, n_paths, 2).clone()
    elapsed = torch.zeros(batch, n_paths, device=DEVICE)
    alive = torch.ones(batch, n_paths, dtype=torch.bool, device=DEVICE)
    max_steps = int(max_time / dt)
    interval = max(1, max_steps // 100)
    noise_scale = sigma * math.sqrt(dt)
    started = time.time()

    for step in range(max_steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current - theta * current * dt + noise_scale * torch.randn_like(current)
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= R**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == max_steps:
            print(f"MC step {step + 1:6d}/{max_steps} | alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s", flush=True)

    if alive.any():
        print(f"Censored at {max_time:g}: {int(alive.sum())}/{alive.numel()} paths")
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


## 5. Editable models and differential operators

`CoordinateNetwork` is the supplied base model, and `make_base_network()` is the single factory used for every method. Edit that class, or return any other `nn.Module` from the factory, to use convolutional features, Fourier features, residual blocks, or another architecture. The only interface requirement is one scalar output per coordinate. `OUModel` then applies the selected trial function and adaptive loss scales; it does not restrict the internal architecture.


In [ ]:
METHOD_SPECS = OrderedDict([
    ("standard", {"power": None, "losses": ("pde", "data", "bc")}),
    ("adapted", {"power": 1.0, "losses": ("pde", "data")}),
    ("over", {"power": 1.5, "losses": ("pde", "data")}),
    ("under", {"power": 0.5, "losses": ("pde", "data")}),
    ("variational", {"power": None, "losses": ("var", "data", "bc")}),
])


def smooth_distance(x):
    return (R**2 - x.square().sum(dim=1)) / (2 * R)


class CoordinateNetwork(nn.Module):
    """Supplied default; replace this class or make_base_network() freely."""

    def __init__(self):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers.extend((nn.Linear(width, width), nn.Tanh()))
        layers.append(nn.Linear(width, 1))
        self.layers = nn.Sequential(*layers)
        if initialization_gain is not None:
            self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight, gain=initialization_gain)
            nn.init.constant_(module.bias, initialization_gain)

    def forward(self, points):
        return self.layers(points)


def make_base_network():
    """Return a fresh scalar network for one method.

    Replace the body with any nn.Module whose output has shape [n] or [n, 1].
    """
    return CoordinateNetwork()


class OUModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        self.power = METHOD_SPECS[method]["power"]
        self.network = make_base_network()
        initial = {name: -0.5 for name in METHOD_SPECS[method]["losses"]}
        if method == "variational":
            initial["bc"] = 3.0  # recorded OU Nitsche initialization
        self.log_scales = nn.ParameterDict({name: nn.Parameter(torch.tensor(value)) for name, value in initial.items()})

    def forward(self, x):
        raw = self.network(x)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (x.shape[0], 1):
            raise ValueError("The base network must return shape [n] or [n, 1].")
        if self.power is None:
            return raw
        return raw * smooth_distance(x).clamp_min(0).pow(self.power).unsqueeze(1)


def strong_residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    generator = -theta * (x * grad).sum(dim=1) + 0.5 * sigma_squared * (tau_xx + tau_yy)
    return generator + 1.0


### Variational/Nitsche residual

Write $A=\sigma^2/2$. Multiplying $-\mathcal Lu=1$ by a test function $\phi$ and integrating the diffusion term by parts gives

$$
A\int_\Omega\nabla u\cdot\nabla\phi\,dx
+\theta\int_\Omega(x\cdot\nabla u)\phi\,dx
-A\int_{\partial\Omega}(\nabla u\cdot n)\phi\,ds
=\int_\Omega\phi\,dx.
$$

The symmetric Nitsche term is $-A\int_{\partial\Omega}u\,\nabla\phi\cdot n\,ds$; the raw penalty $\int_{\partial\Omega}u\phi\,ds$ is the separately weighted boundary component. The notebook uses 50 Gaussian tests

$$\phi_k(x)=\exp\!\left(-\frac{|x-c_k|^2}{2s^2}\right),\qquad s=R/5,$$

and estimates the integrals by sample averages.


In [ ]:
n_test_functions = 50
test_length = R / 5.0
test_centers = sample_disk(n_test_functions)


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]

    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * R**2
    diffusion = 0.5 * sigma_squared * (grad_u[:, None, :] * grad_phi).sum(dim=2)
    drift = theta * (interior * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion + drift).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal = boundary / R
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    arclength = 2 * math.pi * R
    flux_u = (grad_u_b * normal).sum(dim=1)
    flux_phi = (grad_phi_b * normal[:, None, :]).sum(dim=2)
    weak = weak - 0.5 * sigma_squared * (flux_u[:, None] * phi_b).mean(dim=0) * arclength
    symmetry = -0.5 * sigma_squared * (u_b[:, None] * flux_phi).mean(dim=0) * arclength
    raw_boundary = (u_b[:, None] * phi_b).mean(dim=0) * arclength
    return weak + symmetry, raw_boundary


def adaptive_objective(model, components):
    return sum(torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name] for name, value in components.items())


def component_losses(model, method, interior, boundary, data_x, data_y):
    data_loss = (model(data_x).squeeze(1) - data_y).square().mean()
    if method == "variational":
        weak, raw_boundary = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data_loss, "bc": raw_boundary.square().mean()}
    losses = {"pde": strong_residual(model, interior).square().mean(), "data": data_loss}
    if method == "standard":
        losses["bc"] = model(boundary).square().mean()
    return losses


In [ ]:
def run_model_smoke_check():
    interior = sample_disk(8)
    boundary = sample_boundary(8)
    data_x = sample_disk(6)
    data_y = torch.zeros(6, device=DEVICE)
    for method in METHOD_SPECS:
        model = OUModel(method).to(DEVICE)
        components = component_losses(model, method, interior, boundary, data_x, data_y)
        objective = adaptive_objective(model, components)
        if not torch.isfinite(objective):
            raise RuntimeError(f"{method} smoke check produced a non-finite objective")
        objective.backward()
    print("Model smoke check passed for all five methods.")


if RUN_SMOKE_CHECK:
    run_model_smoke_check()


## 6. Shared sampling and Monte Carlo labels

One fresh run creates one label set and one fixed evaluation set, shared by the selected methods. The run folder is unique. Re-running this cell in the same folder stops instead of silently replacing saved data. Resume and evaluation load the saved arrays and check their settings.

In [ ]:
data_path = output_root / f"ou_{RUN_SCALE}_shared_data.pt"
data_settings = {
    "format_version": 2,
    "seed": SEED, "radius": R, "theta": theta, "sigma_squared": sigma_squared,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "evaluation_interior_count": pool_size,
    "evaluation_boundary_count": max(1, pool_size // 4),
}

if data_path.exists():
    if RUN_MODE == "new":
        raise FileExistsError(
            f"{data_path} already exists. Start a new run from Section 3; "
            "saved data will not be overwritten."
        )
    shared = torch.load(data_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(shared, data_settings, data_path)
    x_data, tau_data = shared["x"].to(DEVICE), shared["tau"].to(DEVICE)
    data_censored = shared["censored"].to(DEVICE)
    evaluation_interior = shared["evaluation_interior"].to(DEVICE)
    evaluation_boundary = shared["evaluation_boundary"].to(DEVICE)
    print("Loaded shared labels and evaluation points:", data_path)
else:
    if RUN_MODE != "new":
        raise FileNotFoundError(f"Saved shared data are missing from {output_root}.")
    x_data = sample_disk(data_anchors)
    tau_data, data_censored = monte_carlo_exit_time(x_data, paths_per_anchor, mc_dt, mc_max_time)
    evaluation_interior = sample_disk(pool_size)
    evaluation_boundary = sample_boundary(max(1, pool_size // 4))
    shared = {
        "x": x_data.cpu(), "tau": tau_data.cpu(), "censored": data_censored.cpu(),
        "evaluation_interior": evaluation_interior.cpu(),
        "evaluation_boundary": evaluation_boundary.cpu(),
        "settings": data_settings,
    }
    temporary_data_path = data_path.with_name(data_path.name + ".tmp")
    torch.save(shared, temporary_data_path)
    temporary_data_path.replace(data_path)
    print("Saved shared labels and evaluation points:", data_path)

# Reset the training stream after label generation so new and resumed runs
# initialize their first model from the same recorded training seed.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Shared labels: {len(x_data)} anchors × {paths_per_anchor} paths")
print(f"Censored fraction: mean={float(data_censored.mean()):.2%}, max={float(data_censored.max()):.2%}")

## 7. Adam, checkpointing, and optional L-BFGS

Each method writes `last.pt` for resumption, `best_adam.pt` for the lowest corrected weighted validation objective, and `final.pt` after optional L-BFGS. Checkpoints are written atomically and include the random-number-generator state, so a resumed method continues its saved run. A completed run also saves per-epoch metrics as CSV.

Set `RUN_MODE = "evaluate"`, paste the run folder into `EXISTING_RUN_DIR`, and set `RUN_TRAINING = False` to load completed models without retraining. Evaluation writes `numerical_results.npz`, `numerical_metrics.json`, and comparison plots to that run folder.

In [ ]:
def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    cuda_state = state.get("cuda")
    if cuda_state is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([item.cpu() for item in cuda_state])


def make_scheduler(optimizer):
    step = max(1, adam_epochs // number_of_decays)
    gamma = math.exp(math.log(final_learning_rate_factor) / number_of_decays)
    return torch.optim.lr_scheduler.StepLR(optimizer, step_size=step, gamma=gamma)


def detached_row(components):
    return {name: float(value.detach()) for name, value in components.items()}


def train_method(method):
    run_dir = output_root / f"{RUN_SCALE}_{method}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = OUModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = make_scheduler(optimizer)
    history, start_epoch, best_score = [], 0, math.inf
    last_path, best_path = run_dir / "last.pt", run_dir / "best_adam.pt"
    final_path = run_dir / "final.pt"
    checkpoint_settings = {**experiment_settings, "method": method}
    final_settings = {
        **checkpoint_settings, "lbfgs": RUN_LBFGS,
        "lbfgs_iterations": lbfgs_iterations,
    }

    if RESUME and final_path.exists():
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, final_settings, final_path)
        if "rng_state" not in state:
            raise ValueError(
                f"{final_path} predates saved RNG state and cannot be resumed faithfully. "
                "Use RUN_MODE='evaluate' for its model, or start a new run."
            )
        model.load_state_dict(state["model"])
        restore_rng_state(state["rng_state"])
        return model, pd.DataFrame(state["history"])

    if not RESUME and any(path.exists() for path in (last_path, best_path, final_path)):
        raise FileExistsError(
            f"Training files already exist in {run_dir}. Choose RUN_MODE='resume' "
            "with this run folder, or start a new run."
        )

    if RESUME and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, checkpoint_settings, last_path)
        if "rng_state" not in state:
            raise ValueError(
                f"{last_path} predates saved RNG state and cannot be resumed faithfully. "
                "Use RUN_MODE='evaluate' for its final model, or start a new run."
            )
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        restore_rng_state(state["rng_state"])
        print(f"{method}: resuming after epoch {start_epoch}")
    elif RESUME and best_path.exists():
        raise FileNotFoundError(
            f"{run_dir} has a best checkpoint but no restart checkpoint. "
            "Start a new run or evaluate a completed final.pt."
        )

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        model.train()
        optimizer.zero_grad()
        components = component_losses(model, method, sample_disk(batch_size), sample_boundary(batch_size), x_data, tau_data)
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        validation = component_losses(model, method, evaluation_interior, evaluation_boundary, x_data, tau_data)
        score = float(adaptive_objective(model, validation).detach())
        row = {"epoch": epoch, "objective": float(objective.detach()), "checkpoint_objective": score, **detached_row(components)}
        history.append(row)
        if score < best_score:
            best_score = score
            atomic_torch_save(
                {"model": model.state_dict(), "epoch": epoch, "score": score, "settings": checkpoint_settings},
                best_path,
            )

        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{key} {value:.3e}" for key, value in detached_row(components).items())
            print(f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | {time.time() - started:.1f}s", flush=True)
        if epoch % checkpoint_every == 0 or epoch == 1 or epoch == adam_epochs:
            atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "history": history,
                "epoch": epoch, "best_score": best_score,
                "settings": checkpoint_settings, "rng_state": capture_rng_state(),
            }, last_path)

    best_state = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_state["model"])
    if RUN_LBFGS:
        fixed_interior = sample_disk(pool_size)
        fixed_boundary = sample_boundary(pool_size)
        tolerance = 1e-9 if method in {"standard", "variational"} else 1e-10
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iterations, history_size=1000, tolerance_grad=tolerance, tolerance_change=tolerance, line_search_fn="strong_wolfe")
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            lbfgs.zero_grad()
            parts = component_losses(model, method, fixed_interior, fixed_boundary, x_data, tau_data)
            loss = adaptive_objective(model, parts)
            loss.backward()
            if calls % 100 == 0:
                print(method, "L-BFGS call", calls, detached_row(parts), flush=True)
            return loss

        lbfgs.step(closure)
    atomic_torch_save({
        "model": model.state_dict(), "history": history, "method": method,
        "scale": RUN_SCALE, "settings": final_settings,
        "rng_state": capture_rng_state(),
    }, final_path)
    return model, pd.DataFrame(history)

In [ ]:
trained = {}
histories = {}
if RUN_TRAINING and LOAD_SAVED_MODELS:
    raise ValueError("Choose training or evaluation, not both.")
if RUN_TRAINING:
    for method in METHODS:
        if method not in METHOD_SPECS:
            raise ValueError(f"Unknown method: {method}")
        trained[method], histories[method] = train_method(method)
    print("Training complete:", ", ".join(trained))
elif LOAD_SAVED_MODELS:
    for method in METHODS:
        run_dir = output_root / f"{RUN_SCALE}_{method}"
        final_path = run_dir / "final.pt"
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        expected = {
            **experiment_settings, "method": method, "lbfgs": RUN_LBFGS,
            "lbfgs_iterations": lbfgs_iterations,
        }
        require_matching_settings(state, expected, final_path)
        model = OUModel(method).to(DEVICE)
        model.load_state_dict(state["model"])
        trained[method] = model
        histories[method] = pd.DataFrame(state["history"])
    print("Loaded final models:", ", ".join(trained))
else:
    print("Set RUN_TRAINING = True to train, or RUN_MODE='evaluate' to load a saved run.")

if trained:
    for method, history in histories.items():
        run_dir = output_root / f"{RUN_SCALE}_{method}"
        run_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = run_dir / "training_metrics.csv"
        temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
        history.to_csv(temporary_metrics_path, index=False)
        temporary_metrics_path.replace(metrics_path)
    print("Per-epoch training metrics saved under:", output_root)

## 8. Evaluation

The following cells reproduce the original empirical relative $L^1$, $L^2$, $H^1$, and $H^2$ calculations on a Cartesian grid. The derivative norms use finite differences on the zero-filled disk grid, matching the historical evaluation convention.


In [ ]:
def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        prediction = model(torch.as_tensor(grid_points, device=DEVICE)).squeeze(1).cpu().numpy()
    predicted_grid = np.zeros_like(gx)
    predicted_grid[disk_mask] = prediction
    truth = reference_grid[disk_mask]
    spacing = 2 * R / (grid_n - 1)
    error_grid = predicted_grid - reference_grid
    error_y, error_x = np.gradient(error_grid, spacing, spacing)
    error_xx = np.gradient(error_x, spacing, axis=1)
    error_yy = np.gradient(error_y, spacing, axis=0)
    error_xy = np.gradient(error_y, spacing, axis=1)
    truth_y, truth_x = np.gradient(reference_grid, spacing, spacing)
    truth_xx = np.gradient(truth_x, spacing, axis=1)
    truth_yy = np.gradient(truth_y, spacing, axis=0)
    truth_xy = np.gradient(truth_y, spacing, axis=1)
    rel_l1 = np.mean(np.abs(prediction - truth)) / np.mean(np.abs(truth))
    rel_l2 = np.sqrt(np.mean((prediction - truth)**2) / np.mean(truth**2))
    rel_h1 = np.sqrt(np.mean(error_grid[disk_mask]**2 + error_x[disk_mask]**2 + error_y[disk_mask]**2) / np.mean(reference_grid[disk_mask]**2 + truth_x[disk_mask]**2 + truth_y[disk_mask]**2))
    rel_h2 = np.sqrt(np.mean(error_grid[disk_mask]**2 + error_x[disk_mask]**2 + error_y[disk_mask]**2 + error_xx[disk_mask]**2 + error_yy[disk_mask]**2 + 2 * error_xy[disk_mask]**2) / np.mean(reference_grid[disk_mask]**2 + truth_x[disk_mask]**2 + truth_y[disk_mask]**2 + truth_xx[disk_mask]**2 + truth_yy[disk_mask]**2 + 2 * truth_xy[disk_mask]**2))
    return predicted_grid, {"L1 relative": float(rel_l1), "L2 relative": float(rel_l2), "H1 relative": float(rel_h1), "H2 relative": float(rel_h2)}


if trained:
    figures = {"reference": reference_grid}
    metrics = {}
    error_fields = {}
    for name, model in trained.items():
        figures[name], metrics[name] = evaluate_model(model)
        error_fields[name] = figures[name] - reference_grid

    display(pd.DataFrame(metrics).T)

    result_arrays = {
        "gx": gx,
        "gy": gy,
        "grid_points": grid_points,
        "grid_spacing": np.asarray(2 * R / (grid_n - 1)),
        "grid_size": np.asarray(grid_n),
        "disk_mask": disk_mask,
        "reference_grid": reference_grid,
        "monte_carlo_anchors": x_data.detach().cpu().numpy(),
        "monte_carlo_exit_times": tau_data.detach().cpu().numpy(),
        "monte_carlo_censored_fraction": data_censored.detach().cpu().numpy(),
        "evaluation_interior": evaluation_interior.detach().cpu().numpy(),
        "evaluation_boundary": evaluation_boundary.detach().cpu().numpy(),
        "run_seed": np.asarray(SEED, dtype=np.uint64),
    }
    for name, prediction in figures.items():
        if name != "reference":
            result_arrays[f"prediction_{name}"] = prediction
            result_arrays[f"signed_error_{name}"] = error_fields[name]
    for name, history in histories.items():
        for column in history.select_dtypes(include=[np.number]).columns:
            result_arrays[f"training_{name}_{column}"] = history[column].to_numpy()

    results_path = output_root / "numerical_results.npz"
    temporary_results_path = results_path.with_name(results_path.name + ".tmp")
    with temporary_results_path.open("wb") as handle:
        np.savez_compressed(handle, **result_arrays)
    temporary_results_path.replace(results_path)

    error_limit = max(
        float(np.max(np.abs(values[disk_mask])))
        for values in error_fields.values()
    )
    error_limit = max(error_limit, np.finfo(float).eps)
    solution_values = [figures[name][disk_mask] for name in figures]
    solution_min = min(float(values.min()) for values in solution_values)
    solution_max = max(float(values.max()) for values in solution_values)

    metrics_record = {
        "run_id": output_root.name,
        "seed": SEED,
        "settings": experiment_settings,
        "methods": list(trained),
        "metrics": metrics,
        "solution_color_limits": [solution_min, solution_max],
        "absolute_error_color_limits": [0.0, error_limit],
        "arrays_file": results_path.name,
        "training_metrics_files": {
            name: f"{RUN_SCALE}_{name}/training_metrics.csv" for name in histories
        },
    }
    metrics_path = output_root / "numerical_metrics.json"
    temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
    temporary_metrics_path.write_text(json.dumps(metrics_record, indent=2) + "\n", encoding="utf-8")
    temporary_metrics_path.replace(metrics_path)

    solution_levels = np.linspace(solution_min, solution_max, 51)
    solution_figure, solution_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(solution_axes.flat, figures.items()):
        plot = axis.contourf(
            gx, gy, np.ma.masked_where(~disk_mask, values),
            levels=solution_levels, vmin=solution_min, vmax=solution_max, extend="both",
        )
        axis.set_title(name)
        axis.set_aspect("equal")
        solution_figure.colorbar(plot, ax=axis)
    for axis in solution_axes.flat[len(figures):]:
        axis.axis("off")
    solution_figure.suptitle("OU mean escape time: solution fields")
    solution_figure.savefig(output_root / "solution_fields.png", dpi=180)
    plt.show()

    error_levels = np.linspace(0.0, error_limit, 51)
    error_figure, error_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(error_axes.flat, error_fields.items()):
        plot = axis.contourf(
            gx, gy, np.ma.masked_where(~disk_mask, np.abs(values)),
            levels=error_levels, vmin=0.0, vmax=error_limit, cmap="magma", extend="max",
        )
        axis.set_title(f"{name}: absolute error")
        axis.set_aspect("equal")
        error_figure.colorbar(plot, ax=axis)
    for axis in error_axes.flat[len(error_fields):]:
        axis.axis("off")
    error_figure.suptitle(f"OU mean escape time: shared absolute-error scale [0, {error_limit:.3g}]")
    error_figure.savefig(output_root / "absolute_error_fields.png", dpi=180)
    plt.show()

    print("Numerical arrays, metrics, and plots saved under:", output_root)

The table below records the historical comparison from the original result notebook. Those older runs did not retain enough raw data to regenerate every metric. Runs made with this notebook now save the reference grid, predictions, signed errors, Monte Carlo labels, evaluation points, and per-epoch metrics so the figures and additional metrics can be recomputed without retraining.

## Adapting the method to another PDE

Keep the training and reporting flow, and replace the problem-specific pieces: the drift and diffusion in `strong_residual`, the interior and boundary samplers, the boundary factor in `smooth_distance`, and the reference or Monte Carlo simulator. Check the sign convention by deriving $-\mathcal L u=1$ before editing the residual. `make_base_network()` accepts any scalar PyTorch architecture; the boundary-adapted wrapper and method-specific losses remain separate from that choice.


## References

- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- H. Kersting, A. Orvieto, F. Proske, and A. Lucchi, [“Mean first exit times of Ornstein–Uhlenbeck processes in high-dimensional spaces”](https://arxiv.org/abs/2208.04029), *J. Phys. A* 56 (2023), 215003.
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- Y. Liao and P. Ming, [“Deep Nitsche Method”](https://doi.org/10.4208/cicp.OA-2020-0219), *Commun. Comput. Phys.* 29 (2021).
- A. Kendall, Y. Gal, and R. Cipolla, [“Multi-task learning using uncertainty to weigh losses”](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html), CVPR 2018.
